In [1]:
# Project paths: keep every project file in the same folder as this notebook.
# Open/run the notebook with that folder as the Jupyter working directory.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
print("Project folder:", PROJECT_DIR)
print("All input files and generated outputs will be read from / saved to this folder.")

# Import required libraries

import torch
import transformers
import pandas as pd
import numpy as np
import sklearn
import plotly
import datasets

# Check library versions

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

# Check computing device

device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nComputing device:", device)
print("Environment is ready!")

/home/ely/miniconda3/envs/vis4ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.15.0+cu132
Transformers: 5.22.2
Pandas: 3.1.1
NumPy: 2.4.6

Computing device: cpu
Environment is ready!


In [2]:
from transformers import pipeline

# Selected model
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Load sentiment analysis model
classifier = pipeline(
    "text-classification",
    model=model_name,
    device=-1
)

print("DistilBERT loaded successfully!")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6192.17it/s]


DistilBERT loaded successfully!


In [3]:
# First prediction experiment

test_sentences = [
    "I love this movie.",
    "I hate this movie.",
    "This movie is not bad.",
    "Great! Another problem to deal with."
]

# Get model predictions
predictions = classifier(test_sentences)

# Display results
for sentence, result in zip(test_sentences, predictions):
    print("Text:", sentence)
    print("Prediction:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 50)

Text: I love this movie.
Prediction: POSITIVE
Confidence: 0.9999
--------------------------------------------------
Text: I hate this movie.
Prediction: NEGATIVE
Confidence: 0.9997
--------------------------------------------------
Text: This movie is not bad.
Prediction: POSITIVE
Confidence: 0.9996
--------------------------------------------------
Text: Great! Another problem to deal with.
Prediction: POSITIVE
Confidence: 0.997
--------------------------------------------------


In [4]:
# Organize predictions into a DataFrame

df = pd.DataFrame({
    "text": test_sentences,
    "prediction": [p["label"] for p in predictions],
    "confidence": [p["score"] for p in predictions],
    "category": [
        "simple_positive",
        "simple_negative",
        "negation",
        "sarcasm_candidate"
    ]
})

# Display the dataset
display(df)

,text,prediction,confidence,category
0,I love this movie.,POSITIVE,0.999874,simple_positive
1,I hate this movie.,NEGATIVE,0.999687,simple_negative
2,This movie is not bad.,POSITIVE,0.999604,negation
3,Great! Another problem to deal with.,POSITIVE,0.997015,sarcasm_candidate


In [5]:
from datasets import load_dataset

# Load SST-2 validation dataset
sst2_dataset = load_dataset(
    "stanfordnlp/sst2",
    split="validation"
)

# Convert dataset to Pandas
sst2_df = sst2_dataset.to_pandas()

# Rename sentence column
sst2_df = sst2_df.rename(
    columns={"sentence": "text"}
)

# Convert numeric labels to readable labels
sst2_df["true_label"] = sst2_df["label"].map({
    0: "NEGATIVE",
    1: "POSITIVE"
})

# Check dataset
print("Dataset shape:", sst2_df.shape)
print("\nColumns:")
print(sst2_df.columns.tolist())

display(sst2_df.head())

Dataset shape: (872, 4)

Columns:
['idx', 'text', 'label', 'true_label']


,idx,text,label,true_label
0,0,it 's a charming and often affecting journey .,1,POSITIVE
1,1,unflinchingly bleak and desperate,0,NEGATIVE
2,2,allows us to hope that nolan is poised to emba...,1,POSITIVE
3,3,"the acting , costumes , music , cinematography...",1,POSITIVE
4,4,"it 's slow -- very , very slow .",0,NEGATIVE


In [6]:
# Analyze the dataset structure

print("Total sentences:", len(sst2_df))

# Count positive and negative sentences
label_counts = (
    sst2_df["true_label"]
    .value_counts()
    .reset_index()
)

label_counts.columns = ["sentiment", "count"]

label_counts["percentage"] = (
    label_counts["count"] / len(sst2_df) * 100
).round(2)

display(label_counts)

# Check missing and duplicate texts
print("Missing texts:", sst2_df["text"].isna().sum())
print("Duplicate texts:", sst2_df["text"].duplicated().sum())

# Calculate sentence length
sst2_df["word_count"] = (
    sst2_df["text"].str.split().str.len()
)

print("\nSentence length statistics:")
display(sst2_df["word_count"].describe().round(2))

Total sentences: 872


,sentiment,count,percentage
0,POSITIVE,444,50.92
1,NEGATIVE,428,49.08


Missing texts: 0
Duplicate texts: 0

Sentence length statistics:


count    872.00
mean      19.55
std        8.76
min        2.00
25%       13.00
50%       19.00
75%       26.00
max       47.00
Name: word_count, dtype: float64

In [7]:
# Run DistilBERT on the complete SST-2 validation dataset

texts = sst2_df["text"].tolist()

sst2_predictions = classifier(
    texts,
    batch_size=16,
    truncation=True
)

# Create a copy of the original dataset
results_df = sst2_df.copy()

# Store model outputs
results_df["prediction"] = [
    p["label"] for p in sst2_predictions
]

results_df["confidence"] = [
    p["score"] for p in sst2_predictions
]

# Compare predictions with ground truth
results_df["correct"] = (
    results_df["true_label"] == results_df["prediction"]
)

# Display basic results
print("Total predictions:", len(results_df))

print(
    "Correct predictions:",
    results_df["correct"].sum()
)

print(
    "Incorrect predictions:",
    (~results_df["correct"]).sum()
)

print(
    "Accuracy:",
    round(results_df["correct"].mean() * 100, 2),
    "%"
)

display(results_df.head(10))

Total predictions: 872
Correct predictions: 794
Incorrect predictions: 78
Accuracy: 91.06 %


,idx,text,label,true_label,word_count,prediction,confidence,correct
0,0,it 's a charming and often affecting journey .,1,POSITIVE,9,POSITIVE,0.999884,True
1,1,unflinchingly bleak and desperate,0,NEGATIVE,4,NEGATIVE,0.998970,True
2,2,allows us to hope that nolan is poised to emba...,1,POSITIVE,20,POSITIVE,0.999640,True
3,3,"the acting , costumes , music , cinematography...",1,POSITIVE,20,POSITIVE,0.999699,True
4,4,"it 's slow -- very , very slow .",0,NEGATIVE,9,NEGATIVE,0.999689,True
5,5,although laced with humor and a few fanciful t...,1,POSITIVE,21,POSITIVE,0.999829,True
6,6,a sometimes tedious film .,0,NEGATIVE,5,NEGATIVE,0.999505,True
7,7,or doing last year 's taxes with your ex-wife .,0,NEGATIVE,10,NEGATIVE,0.991760,True
8,8,you do n't have to know about music to appreci...,1,POSITIVE,20,POSITIVE,0.992671,True
9,9,"in exactly 89 minutes , most of which passed a...",0,NEGATIVE,33,NEGATIVE,0.999251,True


In [8]:
from sklearn.metrics import confusion_matrix

# Calculate confusion matrix
labels = ["NEGATIVE", "POSITIVE"]

cm = confusion_matrix(
    results_df["true_label"],
    results_df["prediction"],
    labels=labels
)

# Display as a readable table
cm_df = pd.DataFrame(
    cm,
    index=["Actual NEGATIVE", "Actual POSITIVE"],
    columns=["Predicted NEGATIVE", "Predicted POSITIVE"]
)

display(cm_df)

,Predicted NEGATIVE,Predicted POSITIVE
Actual NEGATIVE,381,47
Actual POSITIVE,31,413


In [9]:
# Extract all incorrect predictions

errors_df = results_df[
    results_df["correct"] == False
].copy()

# Identify error type
errors_df["error_type"] = np.where(
    errors_df["true_label"] == "NEGATIVE",
    "False Positive",
    "False Negative"
)

# Count each error type
print("Error type distribution:")
display(errors_df["error_type"].value_counts())

# Inspect the 15 highest-confidence errors
high_conf_errors = errors_df.sort_values(
    "confidence",
    ascending=False
)

display(
    high_conf_errors[
        [
            "text",
            "true_label",
            "prediction",
            "confidence",
            "word_count",
            "error_type"
        ]
    ].head(15)
)

Error type distribution:


error_type
False Positive    47
False Negative    31
Name: count, dtype: int64

,text,true_label,prediction,confidence,word_count,error_type
791,it 's somewhat clumsy and too lethargically pa...,NEGATIVE,POSITIVE,0.999791,36,False Positive
544,although huppert 's intensity and focus has a ...,NEGATIVE,POSITIVE,0.999657,21,False Positive
183,"the lower your expectations , the more you 'll...",NEGATIVE,POSITIVE,0.999640,12,False Positive
646,"the longer the movie goes , the worse it gets ...",NEGATIVE,POSITIVE,0.999624,23,False Positive
93,if steven soderbergh 's ` solaris ' is a failu...,POSITIVE,NEGATIVE,0.999392,16,False Negative
519,moretti 's compelling anatomy of grief and the...,NEGATIVE,POSITIVE,0.999387,15,False Positive
95,this riveting world war ii moral suspense stor...,NEGATIVE,POSITIVE,0.999312,26,False Positive
695,"sit through this one , and you wo n't need a m...",NEGATIVE,POSITIVE,0.999173,26,False Positive
685,no screen fantasy-adventure in recent memory h...,POSITIVE,NEGATIVE,0.998706,16,False Negative
724,it 's everything you do n't go to the movies f...,NEGATIVE,POSITIVE,0.998681,12,False Positive


In [10]:
# Display complete text of high-confidence errors

for _, row in high_conf_errors.head(15).iterrows():

    print("ID:", row["idx"])
    print("Text:", row["text"])
    print("Actual:", row["true_label"])
    print("Predicted:", row["prediction"])
    print("Confidence:", round(row["confidence"], 4))
    print("Error type:", row["error_type"])
    print("-" * 80)

ID: 791
Text: it 's somewhat clumsy and too lethargically paced -- but its story about a mysterious creature with psychic abilities offers a solid build-up , a terrific climax , and some nice chills along the way . 
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9998
Error type: False Positive
--------------------------------------------------------------------------------
ID: 544
Text: although huppert 's intensity and focus has a raw exhilaration about it , the piano teacher is anything but fun . 
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9997
Error type: False Positive
--------------------------------------------------------------------------------
ID: 183
Text: the lower your expectations , the more you 'll enjoy it . 
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9996
Error type: False Positive
--------------------------------------------------------------------------------
ID: 646
Text: the longer the movie goes , the worse it gets , but it 's actually pretty go

In [11]:
from textwrap import fill

# Display the first 5 high-confidence errors

for _, row in high_conf_errors.head(5).iterrows():

    print("ID:", row["idx"])

    print(
        "Text:",
        fill(
            row["text"],
            width=85,
            subsequent_indent="      "
        )
    )

    print("Actual:", row["true_label"])
    print("Predicted:", row["prediction"])
    print("Confidence:", round(row["confidence"], 4))
    print("Error type:", row["error_type"])
    print("=" * 85)

ID: 791
Text: it 's somewhat clumsy and too lethargically paced -- but its story about a mysterious
      creature with psychic abilities offers a solid build-up , a terrific climax ,
      and some nice chills along the way .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9998
Error type: False Positive
ID: 544
Text: although huppert 's intensity and focus has a raw exhilaration about it , the piano
      teacher is anything but fun .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9997
Error type: False Positive
ID: 183
Text: the lower your expectations , the more you 'll enjoy it .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9996
Error type: False Positive
ID: 646
Text: the longer the movie goes , the worse it gets , but it 's actually pretty good in the
      first few minutes .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9996
Error type: False Positive
ID: 93
Text: if steven soderbergh 's ` solaris ' is a failure it is a glorious failure .
Actual: POSITIVE
Pre

In [12]:
# Inspect the next 3 high-confidence errors

from textwrap import fill

for _, row in high_conf_errors.iloc[3:6].iterrows():

    print("ID:", row["idx"])
    print("Text:", fill(row["text"], width=80))
    print("Actual:", row["true_label"])
    print("Predicted:", row["prediction"])
    print("Confidence:", round(row["confidence"], 4))
    print("Error type:", row["error_type"])
    print("=" * 80)

ID: 646
Text: the longer the movie goes , the worse it gets , but it 's actually pretty good
in the first few minutes .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9996
Error type: False Positive
ID: 93
Text: if steven soderbergh 's ` solaris ' is a failure it is a glorious failure .
Actual: POSITIVE
Predicted: NEGATIVE
Confidence: 0.9994
Error type: False Negative
ID: 519
Text: moretti 's compelling anatomy of grief and the difficult process of adapting to
loss .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9994
Error type: False Positive


In [13]:
# Detect explicit contrast markers

contrast_pattern = (
    r"\b(but|although|though|however|yet|whereas|despite|nevertheless)\b"
)

results_df["has_contrast_marker"] = (
    results_df["text"]
    .str.lower()
    .str.contains(contrast_pattern, regex=True, na=False)
)

# Compare errors across the two groups

contrast_analysis = (
    results_df
    .groupby("has_contrast_marker")
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum()),
        accuracy=("correct", "mean")
    )
)

contrast_analysis["error_rate_%"] = (
    contrast_analysis["errors"] /
    contrast_analysis["total"] * 100
).round(2)

contrast_analysis["accuracy_%"] = (
    contrast_analysis["accuracy"] * 100
).round(2)

display(contrast_analysis)

/tmp/ipykernel_126025/1424602164.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(contrast_pattern, regex=True, na=False)


,total,errors,accuracy,error_rate_%,accuracy_%
has_contrast_marker,,,,,
False,731,60,0.917921,8.21,91.79
True,141,18,0.872340,12.77,87.23


In [14]:
from textwrap import fill

# Select sentences containing contrast markers
contrast_df = results_df[
    results_df["has_contrast_marker"]
].copy()

# Separate incorrect and correct predictions
contrast_errors = contrast_df[
    ~contrast_df["correct"]
].sort_values("confidence", ascending=False)

contrast_correct = contrast_df[
    contrast_df["correct"]
].sample(n=6, random_state=42)

# Compare examples
for title, sample in [
    ("INCORRECT PREDICTIONS", contrast_errors.head(6)),
    ("CORRECT PREDICTIONS", contrast_correct)
]:

    print("\n", title)
    print("=" * 80)

    for _, row in sample.iterrows():

        print("ID:", row["idx"])
        print("Text:", fill(row["text"], width=80))
        print("Actual:", row["true_label"])
        print("Predicted:", row["prediction"])
        print("Confidence:", round(row["confidence"], 4))
        print("-" * 80)


 INCORRECT PREDICTIONS
ID: 791
Text: it 's somewhat clumsy and too lethargically paced -- but its story about a
mysterious creature with psychic abilities offers a solid build-up , a terrific
climax , and some nice chills along the way .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9998
--------------------------------------------------------------------------------
ID: 544
Text: although huppert 's intensity and focus has a raw exhilaration about it , the
piano teacher is anything but fun .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9997
--------------------------------------------------------------------------------
ID: 646
Text: the longer the movie goes , the worse it gets , but it 's actually pretty good
in the first few minutes .
Actual: NEGATIVE
Predicted: POSITIVE
Confidence: 0.9996
--------------------------------------------------------------------------------
ID: 92
Text: you wo n't like roger , but you will quickly recognize him .
Actual: NEGATIVE
Predicted: 

In [15]:
# Limit text width for readable tables

pd.set_option("display.max_colwidth", 200)

# Three incorrect predictions
print("INCORRECT PREDICTIONS")

display(
    contrast_errors[
        [
            "idx",
            "text",
            "true_label",
            "prediction",
            "confidence"
        ]
    ].head(3)
)

# Three correct predictions
print("CORRECT PREDICTIONS")

display(
    contrast_correct[
        [
            "idx",
            "text",
            "true_label",
            "prediction",
            "confidence"
        ]
    ]
)

INCORRECT PREDICTIONS


,idx,text,true_label,prediction,confidence
791,791,"it 's somewhat clumsy and too lethargically paced -- but its story about a mysterious creature with psychic abilities offers a solid build-up , a terrific climax , and some nice chills along the w...",NEGATIVE,POSITIVE,0.999791
544,544,"although huppert 's intensity and focus has a raw exhilaration about it , the piano teacher is anything but fun .",NEGATIVE,POSITIVE,0.999657
646,646,"the longer the movie goes , the worse it gets , but it 's actually pretty good in the first few minutes .",NEGATIVE,POSITIVE,0.999624


CORRECT PREDICTIONS


,idx,text,true_label,prediction,confidence
101,101,"his last movie was poetically romantic and full of indelible images , but his latest has nothing going for it .",NEGATIVE,NEGATIVE,0.997521
299,299,"the sort of film that makes me miss hitchcock , but also feel optimistic that there 's hope for popular cinema yet .",POSITIVE,POSITIVE,0.995357
321,321,but it could have been worse .,NEGATIVE,NEGATIVE,0.995226
596,596,"the director knows how to apply textural gloss , but his portrait of sex-as-war is strictly sitcom .",NEGATIVE,NEGATIVE,0.995354
26,26,"the action switches between past and present , but the material link is too tenuous to anchor the emotional connections that purport to span a 125-year divide .",NEGATIVE,NEGATIVE,0.998062
255,255,no one but a convict guilty of some truly heinous crime should have to sit through the master of disguise .,NEGATIVE,NEGATIVE,0.999519


In [16]:
from scipy.stats import fisher_exact

# Build a 2x2 contingency table
summary = results_df.groupby(
    "has_contrast_marker"
)["correct"].agg(
    total="size",
    errors=lambda x: (~x).sum()
)

a = int(summary.loc[True, "errors"])
b = int(summary.loc[True, "total"] - a)

c = int(summary.loc[False, "errors"])
d = int(summary.loc[False, "total"] - c)

table = [[a, b], [c, d]]

odds_ratio, p_value = fisher_exact(
    table,
    alternative="two-sided"
)

contrast_error_rate = a / (a + b)
noncontrast_error_rate = c / (c + d)

print("Contrast error rate:",
      round(contrast_error_rate * 100, 2), "%")

print("Non-contrast error rate:",
      round(noncontrast_error_rate * 100, 2), "%")

print("Odds ratio:", round(odds_ratio, 3))
print("Fisher p-value:", round(p_value, 4))

Contrast error rate: 12.77 %
Non-contrast error rate: 8.21 %
Odds ratio: 1.637
Fisher p-value: 0.1051


In [17]:
import re

markers = [
    "but",
    "although",
    "though",
    "however",
    "yet",
    "whereas",
    "despite",
    "nevertheless"
]

marker_results = []

for marker in markers:

    pattern = rf"\b{re.escape(marker)}\b"

    mask = results_df["text"].str.lower().str.contains(
        pattern,
        regex=True,
        na=False
    )

    subset = results_df[mask]

    total = len(subset)
    errors = (~subset["correct"]).sum()

    marker_results.append({
        "marker": marker,
        "total": total,
        "errors": errors,
        "error_rate_%": round(errors / total * 100, 2)
        if total > 0 else 0
    })

marker_df = pd.DataFrame(marker_results)

display(
    marker_df.sort_values(
        "total",
        ascending=False
    )
)

,marker,total,errors,error_rate_%
0,but,108,15,13.89
6,despite,11,0,0.00
2,though,9,1,11.11
4,yet,8,1,12.50
1,although,6,2,33.33
3,however,2,0,0.00
5,whereas,0,0,0.00
7,nevertheless,0,0,0.00


In [18]:
from scipy.stats import fisher_exact

# Identify sentences containing the word "but"
but_mask = results_df["text"].str.contains(
    r"\bbut\b",
    case=False,
    regex=True,
    na=False
)

but_group = results_df[but_mask]
no_but_group = results_df[~but_mask]

# Calculate errors and correct predictions
a = int((~but_group["correct"]).sum())
b = int(but_group["correct"].sum())

c = int((~no_but_group["correct"]).sum())
d = int(no_but_group["correct"].sum())

# Statistical test
odds_ratio, p_value = fisher_exact(
    [[a, b], [c, d]],
    alternative="two-sided"
)

# Create comparison table
but_analysis = pd.DataFrame({
    "group": ["With BUT", "Without BUT"],
    "total": [a + b, c + d],
    "errors": [a, c],
    "error_rate_%": [
        round(a / (a + b) * 100, 2),
        round(c / (c + d) * 100, 2)
    ]
})

display(but_analysis)

print("Odds ratio:", round(odds_ratio, 3))
print("Fisher p-value:", round(p_value, 4))

,group,total,errors,error_rate_%
0,With BUT,108,15,13.89
1,Without BUT,764,63,8.25


Odds ratio: 1.795
Fisher p-value: 0.0696


In [19]:
# Separate BUT errors and correct predictions

but_errors = but_group[
    ~but_group["correct"]
].copy()

but_correct_sample = but_group[
    but_group["correct"]
].sample(n=15, random_state=42)

# Display both groups
print("ALL 15 INCORRECT BUT SENTENCES")

display(
    but_errors[
        ["idx", "text", "true_label", "prediction", "confidence"]
    ].sort_values("idx")
)

print("15 RANDOMLY SELECTED CORRECT BUT SENTENCES")

display(
    but_correct_sample[
        ["idx", "text", "true_label", "prediction", "confidence"]
    ].sort_values("idx")
)

ALL 15 INCORRECT BUT SENTENCES


,idx,text,true_label,prediction,confidence
92,92,"you wo n't like roger , but you will quickly recognize him .",NEGATIVE,POSITIVE,0.998519
135,135,"pumpkin means to be an outrageous dark satire on fraternity life , but its ambitions far exceed the abilities of writer adam larson broder and his co-director , tony r. abrams , in their feature d...",NEGATIVE,POSITIVE,0.920258
213,213,"this time mr. burns is trying something in the martin scorsese street-realist mode , but his self-regarding sentimentality trips him up again .",NEGATIVE,POSITIVE,0.975896
405,405,"outer-space buffs might love this film , but others will find its pleasures intermittent .",NEGATIVE,POSITIVE,0.770761
411,411,"i do n't mind having my heartstrings pulled , but do n't treat me like a fool .",NEGATIVE,POSITIVE,0.936814
440,440,"this is wild surreal stuff , but brilliant and the camera just kind of sits there and lets you look at this and its like you 're going from one room to the next and none of them have any relation ...",POSITIVE,NEGATIVE,0.718656
446,446,"with its dogged hollywood naturalism and the inexorable passage of its characters toward sainthood , windtalkers is nothing but a sticky-sweet soap .",NEGATIVE,POSITIVE,0.991039
494,494,"it showcases carvey 's talent for voices , but not nearly enough and not without taxing every drop of one 's patience to get to the good stuff .",NEGATIVE,POSITIVE,0.997371
501,501,"harrison 's flowers puts its heart in the right place , but its brains are in no particular place at all .",POSITIVE,NEGATIVE,0.993843
544,544,"although huppert 's intensity and focus has a raw exhilaration about it , the piano teacher is anything but fun .",NEGATIVE,POSITIVE,0.999657


15 RANDOMLY SELECTED CORRECT BUT SENTENCES


,idx,text,true_label,prediction,confidence
20,20,"pumpkin takes an admirable look at the hypocrisy of political correctness , but it does so with such an uneven tone that you never know when humor ends and tragedy begins .",NEGATIVE,NEGATIVE,0.977344
91,91,"hardly a masterpiece , but it introduces viewers to a good charitable enterprise and some interesting real people .",POSITIVE,POSITIVE,0.999817
100,100,"neither parker nor donovan is a typical romantic lead , but they bring a fresh , quirky charm to the formula .",POSITIVE,POSITIVE,0.999808
189,189,"its story may be a thousand years old , but why did it have to seem like it took another thousand to tell it to us ?",NEGATIVE,NEGATIVE,0.999537
214,214,"perceptive in its vision of nascent industrialized world politics as a new art form , but far too clunky , didactic and saddled with scenes that seem simply an ill fit for this movie .",NEGATIVE,NEGATIVE,0.997523
321,321,but it could have been worse .,NEGATIVE,NEGATIVE,0.995226
379,379,tries to add some spice to its quirky sentiments but the taste is all too familiar .,NEGATIVE,NEGATIVE,0.999611
383,383,"a warm but realistic meditation on friendship , family and affection .",POSITIVE,POSITIVE,0.999874
406,406,"maud and roland 's search for an unknowable past makes for a haunting literary detective story , but labute pulls off a neater trick in possession : he makes language sexy .",POSITIVE,POSITIVE,0.999128
508,508,"a poignant and compelling story about relationships , food of love takes us on a bumpy but satisfying journey of the heart .",POSITIVE,POSITIVE,0.999864


In [ ]:
from pathlib import Path

# Combine all BUT errors and random correct examples
review_errors = but_errors.copy()
review_errors["sample_group"] = "Incorrect"

review_correct = but_correct_sample.copy()
review_correct["sample_group"] = "Correct"

review_df = pd.concat(
    [review_errors, review_correct],
    ignore_index=True
)

# Add columns for human annotation
review_df["but_function"] = ""
review_df["first_clause_polarity"] = ""
review_df["second_clause_polarity"] = ""
review_df["annotation_notes"] = ""

# Save the generated review CSV beside this notebook
output_path = PROJECT_DIR / "but_manual_review.csv"

# Save the review dataset
review_df.to_csv(output_path, index=False)

print("Review dataset saved:")
print(output_path)

print("Total examples:", len(review_df))

display(
    review_df[
        [
            "idx",
            "sample_group",
            "text",
            "but_function",
            "first_clause_polarity",
            "second_clause_polarity"
        ]
    ].head()
)

In [ ]:
from pathlib import Path
import pandas as pd

# Load the annotated CSV from the same folder as this notebook
csv_path = PROJECT_DIR / "but_manual_review_annotated.csv"

if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find {csv_path.name} in {PROJECT_DIR}. "
        "Place the annotated CSV beside this notebook and run the cell again."
    )

# Load annotations
annotation_df = pd.read_csv(csv_path)

# Check the file
print("File loaded successfully:", csv_path.name)
print("Number of samples:", len(annotation_df))
print("Number of columns:", len(annotation_df.columns))

print("\nAnnotation status:")
print(annotation_df["annotation_status"].value_counts())

print("\nSample groups:")
print(annotation_df["sample_group"].value_counts())

display(
    annotation_df[
        [
            "idx",
            "sample_group",
            "but_function",
            "first_clause_polarity",
            "second_clause_polarity",
            "annotation_status"
        ]
    ].head(10)
)


In [22]:
# Inspect AI-generated annotation categories

print("BUT FUNCTION CATEGORIES:")

display(
    annotation_df["but_function"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nBUT FUNCTION BY MODEL RESULT:")

display(
    pd.crosstab(
        annotation_df["but_function"],
        annotation_df["sample_group"],
        dropna=False
    )
)

BUT FUNCTION CATEGORIES:


,count
but_function,
Concession / contrast,5
Contrast,3
Contrast / qualification,3
Discourse continuation (antecedent missing),2
Concession / qualification,2
Contrast / criticism of intention vs. execution,1
Contrast between audiences,1
Concession / boundary-setting,1
Contrast (within a mixed evaluation),1



BUT FUNCTION BY MODEL RESULT:


sample_group,Correct,Incorrect
but_function,,
Adjective-level contrast,1,0
Adjective-level contrast / qualification,1,0
Concession / boundary-setting,0,1
Concession / contrast,3,2
Concession / positive counterpoint,1,0
Concession / qualification,1,1
Contrast,1,2
Contrast (within a mixed evaluation),0,1
Contrast / criticism of intention vs. execution,0,1


In [23]:
# Normalize AI-generated BUT function categories

def normalize_but_function(value):

    if pd.isna(value):
        return "Unclear / Missing"

    text = str(value).lower()

    if "fixed" in text:
        return "Fixed Expression"

    elif "discourse continuation" in text:
        return "Discourse / Other"

    elif "concession" in text:
        return "Concession"

    elif "contrast" in text:
        return "Contrast"

    else:
        return "Other / Unclear"


annotation_df["normalized_function"] = (
    annotation_df["but_function"]
    .apply(normalize_but_function)
)

# Compare normalized categories with model results

category_table = pd.crosstab(
    annotation_df["normalized_function"],
    annotation_df["sample_group"]
)

display(category_table)

print("\nTotal examples per category:")
display(
    annotation_df["normalized_function"]
    .value_counts()
)

sample_group,Correct,Incorrect
normalized_function,,
Concession,5,4
Contrast,8,8
Discourse / Other,1,1
Fixed Expression,1,2



Total examples per category:


normalized_function
Contrast             16
Concession            9
Fixed Expression      3
Discourse / Other     2
Name: count, dtype: int64

In [24]:
%pip install "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import plotly.express as px
from pathlib import Path

# Prepare data for visualization
plot_df = (
    annotation_df
    .groupby(
        ["normalized_function", "sample_group"]
    )
    .size()
    .reset_index(name="count")
)

# Create interactive chart
fig = px.bar(
    plot_df,
    x="normalized_function",
    y="count",
    color="sample_group",
    barmode="group",
    text="count",
    title="BUT Function in Correct and Incorrect Predictions",
    labels={
        "normalized_function": "BUT Function",
        "count": "Number of Examples",
        "sample_group": "Model Result"
    },
    hover_data=["count"]
)

fig.update_layout(
    template="plotly_white",
    height=550,
    xaxis_tickangle=-20,
    legend_title="Model Result"
)

fig.update_traces(textposition="outside")

# Save visualization beside this notebook
figures_dir = PROJECT_DIR

fig.write_html(
    str(figures_dir / "but_function_comparison.html")
)

fig.show()

print("Visualization saved successfully!")

In [26]:
%pip install --upgrade nbformat ipython

Note: you may need to restart the kernel to use updated packages.


In [27]:
# Compare error rates using all 872 SST-2 sentences

but_mask = results_df["text"].str.contains(
    r"\bbut\b",
    case=False,
    regex=True,
    na=False
)

results_df["but_group"] = but_mask.map({
    True: 'Contains "but"',
    False: 'Does not contain "but"'
})

# Calculate statistics
error_analysis = (
    results_df
    .groupby("but_group")
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum())
    )
    .reset_index()
)

error_analysis["error_rate"] = (
    error_analysis["errors"] /
    error_analysis["total"] * 100
).round(2)

display(error_analysis)

,but_group,total,errors,error_rate
0,"Contains ""but""",108,15,13.89
1,"Does not contain ""but""",764,63,8.25


In [ ]:
import plotly.express as px

fig = px.bar(
    error_analysis,
    x="but_group",
    y="error_rate",
    text="error_rate",
    hover_data=["total", "errors"],
    title="DistilBERT Error Rate: With BUT vs Without BUT",
    labels={
        "but_group": "Sentence Group",
        "error_rate": "Error Rate (%)"
    }
)

fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Error Rate (%)",
    yaxis_range=[0, 17],
    height=500
)

fig.write_html(
    str(figures_dir / "but_error_rate_comparison.html")
)

fig.show()

print("Visualization saved successfully!")

In [29]:
# Compare model confidence across groups

confidence_summary = (
    results_df
    .groupby(["but_group", "correct"])["confidence"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

confidence_summary["correct"] = (
    confidence_summary["correct"]
    .map({True: "Correct", False: "Incorrect"})
)

display(
    confidence_summary.round(4)
)

,but_group,correct,count,mean,median,minimum,maximum
0,"Contains ""but""",Incorrect,15,0.9329,0.9759,0.7187,0.9998
1,"Contains ""but""",Correct,93,0.9784,0.9992,0.5249,0.9999
2,"Does not contain ""but""",Incorrect,63,0.9473,0.9857,0.6403,0.9996
3,"Does not contain ""but""",Correct,701,0.9884,0.9996,0.5160,0.9999


In [ ]:
import plotly.express as px

# Prepare readable labels
confidence_plot_df = results_df.copy()

confidence_plot_df["prediction_status"] = (
    confidence_plot_df["correct"]
    .map({
        True: "Correct",
        False: "Incorrect"
    })
)

# Create interactive box plot
fig = px.box(
    confidence_plot_df,
    x="but_group",
    y="confidence",
    color="prediction_status",
    points="all",
    hover_data=[
        "text",
        "true_label",
        "prediction",
        "confidence"
    ],
    title="Confidence Distribution by BUT and Prediction Correctness",
    labels={
        "but_group": "Sentence Group",
        "confidence": "Model Confidence",
        "prediction_status": "Prediction Status"
    }
)

fig.update_layout(
    template="plotly_white",
    height=650,
    yaxis_range=[0.5, 1.01]
)

fig.write_html(
    str(figures_dir / "confidence_distribution.html")
)

fig.show()

In [31]:
# Analyze high-confidence errors

thresholds = [0.95, 0.99]

high_conf_summary = []

for threshold in thresholds:

    subset = results_df[
        results_df["confidence"] >= threshold
    ]

    for group, group_df in subset.groupby("but_group"):

        high_conf_summary.append({
            "threshold": threshold,
            "group": group,
            "total": len(group_df),
            "errors": int((~group_df["correct"]).sum()),
            "error_rate_%": round(
                (~group_df["correct"]).mean() * 100, 2
            )
        })

high_conf_df = pd.DataFrame(high_conf_summary)

display(high_conf_df)

,threshold,group,total,errors,error_rate_%
0,0.95,"Contains ""but""",94,8,8.51
1,0.95,"Does not contain ""but""",712,43,6.04
2,0.99,"Contains ""but""",85,7,8.24
3,0.99,"Does not contain ""but""",651,26,3.99


In [32]:
from scipy.stats import fisher_exact

for threshold in [0.95, 0.99]:

    subset = results_df[
        results_df["confidence"] >= threshold
    ]

    but = subset[subset["but_group"] == 'Contains "but"']
    no_but = subset[subset["but_group"] == 'Does not contain "but"']

    # Errors and correct predictions
    table = [
        [
            int((~but["correct"]).sum()),
            int(but["correct"].sum())
        ],
        [
            int((~no_but["correct"]).sum()),
            int(no_but["correct"].sum())
        ]
    ]

    odds_ratio, p_value = fisher_exact(table)

    print(f"\nConfidence threshold: {threshold}")
    print("Odds Ratio:", round(odds_ratio, 3))
    print("Fisher p-value:", round(p_value, 4))


Confidence threshold: 0.95
Odds Ratio: 1.447
Fisher p-value: 0.3652

Confidence threshold: 0.99
Odds Ratio: 2.157
Fisher p-value: 0.0903


In [33]:
# Compare error rates by BUT presence and true sentiment

class_error_analysis = (
    results_df
    .groupby(["but_group", "true_label"])
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum()),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

class_error_analysis["error_rate_%"] = (
    class_error_analysis["errors"] /
    class_error_analysis["total"] * 100
).round(2)

class_error_analysis["accuracy_%"] = (
    class_error_analysis["accuracy"] * 100
).round(2)

display(class_error_analysis)

,but_group,true_label,total,errors,accuracy,error_rate_%,accuracy_%
0,"Contains ""but""",NEGATIVE,57,10,0.824561,17.54,82.46
1,"Contains ""but""",POSITIVE,51,5,0.901961,9.80,90.20
2,"Does not contain ""but""",NEGATIVE,371,37,0.900270,9.97,90.03
3,"Does not contain ""but""",POSITIVE,393,26,0.933842,6.62,93.38


In [ ]:
import plotly.express as px

fig = px.bar(
    class_error_analysis,
    x="true_label",
    y="error_rate_%",
    color="but_group",
    barmode="group",
    text="error_rate_%",
    hover_data=["total", "errors"],
    title="Error Rate by BUT Presence and True Sentiment",
    labels={
        "true_label": "Actual Sentiment",
        "error_rate_%": "Error Rate (%)",
        "but_group": "Sentence Group"
    }
)

fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    yaxis_title="Error Rate (%)"
)

fig.write_html(
    str(figures_dir / "error_rate_by_sentiment.html")
)

fig.show()


In [35]:
# Compare sentence length between BUT groups

length_analysis = (
    results_df
    .groupby("but_group")["word_count"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max"
    )
    .round(2)
)

display(length_analysis)

,count,mean,median,std,minimum,maximum
but_group,,,,,,
"Contains ""but""",108,24.94,25.0,8.75,5,47
"Does not contain ""but""",764,18.79,18.0,8.50,2,46


In [36]:
from scipy.stats import mannwhitneyu

# Get sentence lengths for both groups
but_lengths = results_df[
    results_df["but_group"] == 'Contains "but"'
]["word_count"]

no_but_lengths = results_df[
    results_df["but_group"] == 'Does not contain "but"'
]["word_count"]

# Statistical test
u_statistic, p_value = mannwhitneyu(
    but_lengths,
    no_but_lengths,
    alternative="two-sided"
)

print("Mean length with BUT:",
      round(but_lengths.mean(), 2))

print("Mean length without BUT:",
      round(no_but_lengths.mean(), 2))

print("Mann-Whitney U:", u_statistic)
print("p-value:", round(p_value, 6))

Mean length with BUT: 24.94
Mean length without BUT: 18.79
Mann-Whitney U: 57384.0
p-value: 0.0


In [37]:
# Create sentence-length groups

results_df["length_group"] = pd.cut(
    results_df["word_count"],
    bins=[0, 10, 20, 30, 50],
    labels=[
        "Short (1-10)",
        "Medium (11-20)",
        "Long (21-30)",
        "Very Long (31-50)"
    ]
)

# Compare error rates across length groups

length_error_analysis = (
    results_df
    .groupby("length_group", observed=True)
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum()),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

length_error_analysis["error_rate_%"] = (
    length_error_analysis["errors"] /
    length_error_analysis["total"] * 100
).round(2)

length_error_analysis["accuracy_%"] = (
    length_error_analysis["accuracy"] * 100
).round(2)

display(length_error_analysis)

,length_group,total,errors,accuracy,error_rate_%,accuracy_%
0,Short (1-10),152,11,0.927632,7.24,92.76
1,Medium (11-20),340,32,0.905882,9.41,90.59
2,Long (21-30),275,25,0.909091,9.09,90.91
3,Very Long (31-50),105,10,0.904762,9.52,90.48


In [38]:
# Compare BUT and non-BUT sentences within each length group

controlled_length_analysis = (
    results_df
    .groupby(
        ["length_group", "but_group"],
        observed=True
    )
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum())
    )
    .reset_index()
)

controlled_length_analysis["error_rate_%"] = (
    controlled_length_analysis["errors"]
    / controlled_length_analysis["total"] * 100
).round(2)

display(controlled_length_analysis)

,length_group,but_group,total,errors,error_rate_%
0,Short (1-10),"Contains ""but""",5,2,40.00
1,Short (1-10),"Does not contain ""but""",147,9,6.12
2,Medium (11-20),"Contains ""but""",23,3,13.04
3,Medium (11-20),"Does not contain ""but""",317,29,9.15
4,Long (21-30),"Contains ""but""",51,7,13.73
5,Long (21-30),"Does not contain ""but""",224,18,8.04
6,Very Long (31-50),"Contains ""but""",29,3,10.34
7,Very Long (31-50),"Does not contain ""but""",76,7,9.21


In [39]:
# Compare error rates by length, true sentiment and BUT presence

stratified_analysis = (
    results_df
    .groupby(
        ["length_group", "true_label", "but_group"],
        observed=True
    )
    .agg(
        total=("correct", "size"),
        errors=("correct", lambda x: (~x).sum())
    )
    .reset_index()
)

stratified_analysis["error_rate_%"] = (
    stratified_analysis["errors"]
    / stratified_analysis["total"] * 100
).round(2)

display(
    stratified_analysis.sort_values(
        ["length_group", "true_label", "but_group"]
    )
)

,length_group,true_label,but_group,total,errors,error_rate_%
0,Short (1-10),NEGATIVE,"Contains ""but""",2,0,0.00
1,Short (1-10),NEGATIVE,"Does not contain ""but""",81,6,7.41
2,Short (1-10),POSITIVE,"Contains ""but""",3,2,66.67
3,Short (1-10),POSITIVE,"Does not contain ""but""",66,3,4.55
4,Medium (11-20),NEGATIVE,"Contains ""but""",14,3,21.43
5,Medium (11-20),NEGATIVE,"Does not contain ""but""",151,16,10.60
6,Medium (11-20),POSITIVE,"Contains ""but""",9,0,0.00
7,Medium (11-20),POSITIVE,"Does not contain ""but""",166,13,7.83
8,Long (21-30),NEGATIVE,"Contains ""but""",27,5,18.52
9,Long (21-30),NEGATIVE,"Does not contain ""but""",99,10,10.10


In [43]:
import importlib.util

if importlib.util.find_spec("statsmodels") is not None:
    print("Statsmodels is installed.")
    import statsmodels
    print("Version:", statsmodels.__version__)
else:
    print("Statsmodels is not installed.")

Statsmodels is installed.
Version: 0.15.0


In [41]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [44]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Prepare data for logistic regression
analysis_df = results_df.copy()

# Error = 1 means incorrect prediction
analysis_df["error"] = (
    ~analysis_df["correct"]
).astype(int)

# BUT indicator: 1 = contains BUT, 0 = does not
analysis_df["has_but"] = (
    analysis_df["but_group"] == 'Contains "but"'
).astype(int)

# Scale sentence length: one unit = 10 words
analysis_df["word_count_10"] = (
    analysis_df["word_count"] / 10
)

# Fit logistic regression with robust standard errors
logit_model = smf.logit(
    "error ~ has_but + word_count_10 + C(true_label)",
    data=analysis_df
).fit(
    disp=False,
    cov_type="HC3"
)

# Display statistical results
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:                  error   No. Observations:                  872
Model:                          Logit   Df Residuals:                      868
Method:                           MLE   Df Model:                            3
Date:                Mon, 09 Nov 2026   Pseudo R-squ.:                 0.01439
Time:                        22:24:38   Log-Likelihood:                -258.92
converged:                       True   LL-Null:                       -262.70
Covariance Type:                  HC3   LLR p-value:                   0.05599
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -2.2903      0.299     -7.652      0.000      -2.877      -1.704
C(true_label)[T.POSITIVE]    -0.4915      0.244     -2.019      0.044      -0.969     

In [45]:
# Convert logistic regression coefficients into Odds Ratios

ci = logit_model.conf_int()

odds_ratio_table = pd.DataFrame({
    "Odds Ratio": np.exp(logit_model.params),
    "CI Lower": np.exp(ci[0]),
    "CI Upper": np.exp(ci[1]),
    "P-value": logit_model.pvalues
})

display(odds_ratio_table.round(4))

,Odds Ratio,CI Lower,CI Upper,P-value
Intercept,0.1012,0.0563,0.1820,0.0000
C(true_label)[T.POSITIVE],0.6117,0.3795,0.9858,0.0435
has_but,1.7027,0.8953,3.2383,0.1046
word_count_10,1.0593,0.8050,1.3938,0.6809


In [ ]:
import plotly.graph_objects as go

# Prepare regression results (exclude intercept)
forest_df = odds_ratio_table.drop(
    index="Intercept"
).copy()

forest_df["variable"] = [
    "BUT present",
    "Positive sentiment",
    "Sentence length (per 10 words)"
]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=forest_df["Odds Ratio"],
        y=forest_df["variable"],
        mode="markers",
        marker=dict(size=12),
        error_x=dict(
            type="data",
            symmetric=False,
            array=(
                forest_df["CI Upper"]
                - forest_df["Odds Ratio"]
            ),
            arrayminus=(
                forest_df["Odds Ratio"]
                - forest_df["CI Lower"]
            ),
            visible=True
        ),
        text=[
            f"OR: {r['Odds Ratio']:.3f}<br>"
            f"95% CI: {r['CI Lower']:.3f}–{r['CI Upper']:.3f}<br>"
            f"p-value: {r['P-value']:.4f}"
            for _, r in forest_df.iterrows()
        ],
        hoverinfo="text"
    )
)

fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="red",
    annotation_text="No association (OR = 1)"
)

fig.update_layout(
    title="Factors Associated with DistilBERT Prediction Errors",
    xaxis_title="Odds Ratio (95% Confidence Interval)",
    yaxis_title="Predictor",
    template="plotly_white",
    height=500
)

fig.write_html(
    str(figures_dir / "logistic_regression_forest_plot.html")
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Correct mapping between model variables and labels
variable_names = {
    "has_but": "BUT present",
    "C(true_label)[T.POSITIVE]": "Positive sentiment",
    "word_count_10": "Sentence length (per 10 words)"
}

forest_df = odds_ratio_table.loc[
    list(variable_names.keys())
].copy()

forest_df["variable"] = [
    variable_names[name]
    for name in forest_df.index
]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=forest_df["Odds Ratio"],
        y=forest_df["variable"],
        mode="markers",
        marker=dict(size=12),
        error_x=dict(
            type="data",
            symmetric=False,
            array=forest_df["CI Upper"] - forest_df["Odds Ratio"],
            arrayminus=forest_df["Odds Ratio"] - forest_df["CI Lower"],
            visible=True
        ),
        text=[
            f"Odds Ratio: {r['Odds Ratio']:.3f}<br>"
            f"95% CI: {r['CI Lower']:.3f}–{r['CI Upper']:.3f}<br>"
            f"P-value: {r['P-value']:.4f}"
            for _, r in forest_df.iterrows()
        ],
        hoverinfo="text"
    )
)

fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="red"
)

fig.update_layout(
    title="Factors Associated with DistilBERT Prediction Errors",
    xaxis_title="Odds Ratio (95% Confidence Interval)",
    yaxis_title="Predictor",
    template="plotly_white",
    height=500
)

fig.write_html(
    str(figures_dir / "logistic_regression_forest_plot.html")
)

fig.show()